In [1]:
import spacy

In [3]:
nlp = spacy.load("en_core_web_sm")
text = "Elon Musk founded SpaceX in California."
doc = nlp(text)

for ent in doc.ents:
    print(ent.text, ent.label_)

Elon Musk PERSON
California GPE


In [4]:
spacy.displacy.render(doc, style="ent", jupyter=True)


In [7]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import pipeline

# Load IndoBERT fine-tuned for NER
model_name = "cahya/bert-base-indonesian-NER"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)

# Create pipeline
nlp_ner = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple")

# Example Indonesian text
text = "Presiden Joko Widodo menghadiri acara di Jakarta bersama Menteri Keuangan Sri Mulyani."

# Run NER
entities = nlp_ner(text)
entities

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 22529.33it/s]
BertForTokenClassification LOAD REPORT from: cahya/bert-base-indonesian-NER
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.pooler.dense.bias       | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 
bert.pooler.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'entity_group': 'PER',
  'score': np.float32(0.9901633),
  'word': 'joko widodo',
  'start': 9,
  'end': 20},
 {'entity_group': 'GPE',
  'score': np.float32(0.99606293),
  'word': 'jakarta',
  'start': 41,
  'end': 48},
 {'entity_group': 'NOR',
  'score': np.float32(0.9728811),
  'word': 'menteri keuangan',
  'start': 57,
  'end': 73},
 {'entity_group': 'PER',
  'score': np.float32(0.9960707),
  'word': 'sri mulyani',
  'start': 74,
  'end': 85}]

In [8]:
from spacy import displacy

# Convert to spaCy-style
doc_data = {
    "text": text,
    "ents": [{"start": e["start"], "end": e["end"], "label": e["entity_group"]} for e in entities],
    "title": None,
}

# Visualize inline
displacy.render(doc_data, style="ent", manual=True, jupyter=True)


In [9]:
text2 = "Insiden keracunan MBG di sekolah Ratusan pelajar SMK Negeri 1 Cihampelas di Kabupaten Bandung Barat kembali mendapat paket MBG pada Rabu (24/09). Menu hari itu adalah telur, pecel, dan kentang rebus. BR turut mengonsumsi MBG yang dikirim dari Satuan Pelayanan Pemenuhan Gizi (SPPG) Maju Jaya tersebut. Kalau berdasarkan pengakuan dari teman-temannya beliau (BR) itu mengonsumsi (MBG), ungkap Wali Kelas BR, Imron Komarudin, saat ditemui di SMK Negeri 1 Cihampelas, Jumat (03/10). Nahas, paket MBG pada hari itu mengakibatkan 167 pelajar SMKN 1 Cihampelas keracunan. BR dan 32 teman sekelasnya tidak termasuk di antaranya. Setelah kejadian [keracunan] itu, saya langsung mengumumkan di grup orang tua siswa, jika terdapat gejala segera sampaikan, ujar Imron."

In [ ]:
# --- 🔐 1. Authenticate to Hugging Face ---
from huggingface_hub import login

# Paste your personal access token from https://huggingface.co/settings/tokens
# ⚠️ DO NOT share this token publicly.
hf_token = "xxxxxxxxxxxxxxxxx"  # ← Replace with your token
login(token=hf_token)

In [11]:
# --- 2. Imports ---
import re, json
import spacy
from spacy import displacy
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
from IPython.display import display, HTML

In [12]:
# --- 3. Load your chosen (private or public) model ---

model_name = "cahya/bert-base-indonesian-NER"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)
ner_pipe = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 24371.96it/s]
BertForTokenClassification LOAD REPORT from: cahya/bert-base-indonesian-NER
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.pooler.dense.bias       | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 
bert.pooler.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
print(ner_pipe("Presiden Joko Widodo menghadiri acara di Jakarta bersama Menteri Keuangan Sri Mulyani."))


[{'entity_group': 'PER', 'score': np.float32(0.9901633), 'word': 'joko widodo', 'start': 9, 'end': 20}, {'entity_group': 'GPE', 'score': np.float32(0.99606293), 'word': 'jakarta', 'start': 41, 'end': 48}, {'entity_group': 'NOR', 'score': np.float32(0.9728811), 'word': 'menteri keuangan', 'start': 57, 'end': 73}, {'entity_group': 'PER', 'score': np.float32(0.9960707), 'word': 'sri mulyani', 'start': 74, 'end': 85}]


In [14]:
# Load your chosen (private or public) model
model_name = "cahya/bert-base-indonesian-NER"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)
ner_pipe = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple")

nlp = spacy.blank("xx")

def align_entities(text, entities):
    fixed = []
    pos = 0
    for e in entities:
        word = e.get("word", "").strip()
        if not word:
            continue
        match = re.search(re.escape(word), text[pos:])
        if match:
            start = pos + match.start()
            end = pos + match.end()
            fixed.append({"start": start, "end": end, "label": e["entity_group"]})
            pos = end
    return fixed

def visualize_ner_spacy(text, show_raw=False):
    if not text.strip():
        print("Masukkan teks Bahasa Indonesia terlebih dahulu.")
        return

    entities = ner_pipe(text)
    if show_raw:
        print(json.dumps(entities, indent=2, ensure_ascii=False))

    ents_fixed = align_entities(text, entities)
    if not ents_fixed:
        print("Tidak ditemukan entitas yang bisa dipetakan ke teks.")
        return

    doc = nlp(text)
    spans = []
    for ent in ents_fixed:
        span = doc.char_span(ent["start"], ent["end"], label=ent["label"], alignment_mode="expand")
        if span:
            spans.append(span)
    doc.ents = spans

    html_out = displacy.render(doc, style="ent")
    display(HTML(html_out))



Loading weights: 100%|██████████| 199/199 [00:00<00:00, 20437.48it/s]
BertForTokenClassification LOAD REPORT from: cahya/bert-base-indonesian-NER
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.pooler.dense.bias       | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 
bert.pooler.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [15]:
text3= "Dalam perjalanan ke RSUD Cililin, Nanang melihat kondisi keponakannya sudah pucat kebiru-biruan. Pria berusia 53 tahun itu merasa BR sudah tidak ada. Ternyata pas masuk ke Rumah Sakit Cililin, orang-orang IGD sudah langsung pada ngomong, ini mah pasien DoA (Dead on Arrival). Jadi meninggal sebelum sampai rumah sakit, sebut Nanang. Kepala Pelayanan IGD RSUD Cililin, dr. Dwi Puspitasari Anggita mengatakan, pihaknya menerima BR di IGD RSUD Cililin pada Selasa (30/09) pukul 13.30 WIB."

In [16]:
# Create pipeline
nlp_ner = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple")

# Run NER
entities3 = nlp_ner(text3)
entities3

[{'entity_group': 'FAC',
  'score': np.float32(0.91476494),
  'word': 'rsud cililin',
  'start': 20,
  'end': 32},
 {'entity_group': 'PER',
  'score': np.float32(0.9845901),
  'word': 'nanang',
  'start': 34,
  'end': 40},
 {'entity_group': 'QTY',
  'score': np.float32(0.9604173),
  'word': '53 tahun',
  'start': 110,
  'end': 118},
 {'entity_group': 'PER',
  'score': np.float32(0.9853065),
  'word': 'br',
  'start': 130,
  'end': 132},
 {'entity_group': 'FAC',
  'score': np.float32(0.9409898),
  'word': 'rumah sakit cililin',
  'start': 172,
  'end': 191},
 {'entity_group': 'ORG',
  'score': np.float32(0.6645857),
  'word': 'igd',
  'start': 205,
  'end': 208},
 {'entity_group': 'PER',
  'score': np.float32(0.95531374),
  'word': 'nanang',
  'start': 325,
  'end': 331},
 {'entity_group': 'NOR',
  'score': np.float32(0.5610956),
  'word': 'kepala pelayanan igd',
  'start': 333,
  'end': 353},
 {'entity_group': 'ORG',
  'score': np.float32(0.38639373),
  'word': 'rsud cili',
  'start': 

In [17]:
import spacy
from spacy import displacy
from IPython.display import display, HTML

# Create blank spaCy doc (no NLP model needed)
nlp = spacy.blank("xx")

# Build doc_data manually for displaCy
doc_data = {
    "text": text3,
    "ents": [
        {"start": ent["start"], "end": ent["end"], "label": ent["entity_group"]}
        for ent in entities3
    ],
    "title": "NER Visualization"
}

# Render inside notebook
html = displacy.render(doc_data, style="ent", manual=True, jupyter=False)
display(HTML(html))

In [18]:
text4= "Seorang siswi SMK Negeri 1 Cihampelas di Kabupaten Bandung Barat, Jawa Barat, meninggal dunia, Selasa (30/09). Kematiannya dikaitkan dengan kasus keracunan massal program Makanan Bergizi Gratis (MBG) hampir sepekan sebelumnya."

In [19]:
# Create pipeline
nlp_ner = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple")

# Run NER
entities4 = nlp_ner(text4)
entities4

[{'entity_group': 'ORG',
  'score': np.float32(0.5971875),
  'word': 'smk negeri 1 cihampelas',
  'start': 14,
  'end': 37},
 {'entity_group': 'GPE',
  'score': np.float32(0.9851237),
  'word': 'kabupaten bandung barat',
  'start': 41,
  'end': 64},
 {'entity_group': 'GPE',
  'score': np.float32(0.993293),
  'word': 'jawa barat',
  'start': 66,
  'end': 76},
 {'entity_group': 'DAT',
  'score': np.float32(0.9923915),
  'word': 'selasa ( 30 / 09 )',
  'start': 95,
  'end': 109},
 {'entity_group': 'EVT',
  'score': np.float32(0.385126),
  'word': 'mb',
  'start': 195,
  'end': 197}]

In [20]:
import spacy
from spacy import displacy
from IPython.display import display, HTML

# Create blank spaCy doc (no NLP model needed)
nlp = spacy.blank("xx")

# Build doc_data manually for displaCy
doc_data = {
    "text": text4,
    "ents": [
        {"start": ent["start"], "end": ent["end"], "label": ent["entity_group"]}
        for ent in entities4
    ],
    "title": "NER Visualization"
}

# Render inside notebook
html = displacy.render(doc_data, style="ent", manual=True, jupyter=False)
display(HTML(html))

In [21]:
text5= "Menu hari itu adalah telur, pecel, dan kentang rebus. BR turut mengonsumsi MBG yang dikirim dari Satuan Pelayanan Pemenuhan Gizi (SPPG) Maju Jaya tersebut."

In [22]:
# Create pipeline
nlp_ner = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple")

# Run NER
entities5 = nlp_ner(text5)
entities5

[{'entity_group': 'PRD',
  'score': np.float32(0.9702814),
  'word': 'telur',
  'start': 21,
  'end': 26},
 {'entity_group': 'PRD',
  'score': np.float32(0.611644),
  'word': 'pecel',
  'start': 28,
  'end': 33},
 {'entity_group': 'PRD',
  'score': np.float32(0.9519212),
  'word': 'kentang rebus',
  'start': 39,
  'end': 52},
 {'entity_group': 'PER',
  'score': np.float32(0.76632386),
  'word': 'br',
  'start': 54,
  'end': 56},
 {'entity_group': 'PRD',
  'score': np.float32(0.9387169),
  'word': 'mbg',
  'start': 75,
  'end': 78},
 {'entity_group': 'ORG',
  'score': np.float32(0.8285682),
  'word': 'satuan pelayanan pemenuhan gizi',
  'start': 97,
  'end': 128},
 {'entity_group': 'ORG',
  'score': np.float32(0.6189857),
  'word': 'sppg ) maju jaya',
  'start': 130,
  'end': 145}]

In [23]:
import spacy
from spacy import displacy
from IPython.display import display, HTML

# Create blank spaCy doc (no NLP model needed)
nlp = spacy.blank("xx")

# Build doc_data manually for displaCy
doc_data = {
    "text": text5,
    "ents": [
        {"start": ent["start"], "end": ent["end"], "label": ent["entity_group"]}
        for ent in entities5
    ],
    "title": "NER Visualization"
}

# Render inside notebook
html = displacy.render(doc_data, style="ent", manual=True, jupyter=False)
display(HTML(html))